**对话历史会无限增长**

In [3]:
from dotenv import load_dotenv
load_dotenv()   
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


model = init_chat_model(model="deepseek:deepseek-chat")


agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "long_conversation"}}

# 模拟多轮对话
print("\n模拟 10 轮对话...")
for i in range(1, 11):
    agent.invoke(
        {"messages": [{"role": "user", "content": f"这是第 {i} 轮对话"}]},
        config=config
    )

# 查看消息数量
response = agent.invoke(
    {"messages": [{"role": "user", "content": "总结一下"}]},  
    config=config
)

print(f"\n总消息数: {len(response['messages'])}")
print("(包含用户消息 + AI 回复)")


模拟 10 轮对话...

总消息数: 22
(包含用户消息 + AI 回复)


**SummarizationMiddleware**

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model


model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)


agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,   # 用于生成摘要的模型
            max_tokens_before_summary=200  # 超过 200 tokens 触发摘要
        )
    ]
)

config = {"configurable": {"thread_id": "with_summary"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师",
    "我在北京工作",
    "我喜欢编程和阅读",
    "我最近在学习 AI",
    "请总结一下我的信息"
]

for msg in conversations:
    print(f"\n用户: {msg}")
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    # print(f"Agent: {response}")
    # print(f"Agent: {response['messages'][-1].content[:60]}...")
    print(f"Agent: {response['messages'][-1].content}")

# print(response)


进行多轮对话...

用户: 我叫张三，是工程师
Agent: 张三你好！很高兴认识你！作为工程师，你一定在逻辑思维、解决问题或创造技术方案方面有很多经验。如果有什么技术问题需要讨论，或者想分享你的项目想法，随时可以告诉我。 😊

（如果你愿意，也可以简单说说你具体从事哪个领域的工程工作？比如软件、硬件、机械、土木等等～）

用户: 我在北京工作
Agent: 北京作为中国的科技创新中心，确实有很多工程师发展的机会！无论是互联网、人工智能、硬件研发，还是传统行业的工程领域，北京都有丰富的资源和活跃的生态圈。

你在北京工作的话，可能会接触到很多前沿项目或大型工程。如果工作中遇到技术挑战，或者想聊聊行业趋势、职业发展，我都很乐意一起探讨～ 

（最近北京天气如何？工程师平时工作节奏快，记得多注意休息哦！ 😄）

用户: 我喜欢编程和阅读
Agent: 太棒了！编程和阅读是非常互补的两个爱好——编程是**创造与逻辑的实践**，阅读则是**吸收与灵感的源泉**。这两个习惯结合，往往能让你在技术和思维上都不断突破。

### 💻 关于编程：
- 你主要用哪些语言或技术栈？是偏向后端、前端、算法，还是嵌入式/硬件相关？
- 最近有在做什么有趣的项目，或者在学习什么新框架/工具吗？

### 📚 关于阅读：
- 你喜欢读技术书籍（比如架构、算法、编程思想），还是更偏向非技术类（文学、社科、历史、科幻等）？
- 有没有最近读到的、特别想推荐的书？

---

作为工程师，阅读技术内容能帮你深化专业能力，而跨领域的阅读常常能带来意想不到的灵感（比如科幻对科技的前瞻、历史对系统复杂性的理解）。如果需要书单推荐、技术问题探讨，或者想聊聊某本书的启发，我随时都在～ 

（另外，如果喜欢阅读，北京有很多很棒的书店和图书馆，比如 PageOne、钟书阁、国家图书馆，周末去逛逛或许会有惊喜哦！）

用户: 我最近在学习 AI
Agent: 太棒了！AI 是现在最令人兴奋的领域之一，无论是理论研究还是工程落地都有巨大的探索空间。作为工程师，你的编程和逻辑基础会是非常好的起点。

### 🤖 学习 AI 的常见路径参考：
1. **基础数学**：线性代数、概率统计、微积分（很多框架已封装，但理解原理很重要）。
2. **核心技能**：Python + 常用库（NumPy、Pandas、Matpl

**trim_messages**

In [11]:
from langchain_core.messages import trim_messages

# 模拟一个长对话历史
from langchain_core.messages import HumanMessage, AIMessage

messages = [
    HumanMessage(content="消息 1"),
    AIMessage(content="回复 1"),
    HumanMessage(content="消息 2"),
    AIMessage(content="回复 2"),
    HumanMessage(content="消息 3"),
    AIMessage(content="回复 3"),
    HumanMessage(content="消息 4"),
    AIMessage(content="回复 4"),
]

print(f"\n原始消息数: {len(messages)}")


# 按 token 数裁剪（不严格条数）	max_tokens=N + 合理 token_counter
# 严格保留最后 N 条消息	max_count=N

trimmed = trim_messages(
    messages,
    max_tokens=2,  # 或使用 token 数限制
    strategy="last",  # 保留最后的消息
    token_counter=len  # 简单计数器（实际应该用 token 计数）
)

print(f"修剪后消息数: {len(trimmed)}")
print("\n保留的消息：")
for msg in trimmed:
    # msg.pretty_print()
    print(f"  {msg.__class__.__name__}: {msg.content}")


原始消息数: 8
修剪后消息数: 2

保留的消息：
  HumanMessage: 消息 4
  AIMessage: 回复 4


**客服机器人**

In [12]:
from langchain_core.tools import tool

@tool
def calculator(operation: str, a: float, b: float) -> str:
    """执行数学计算"""
    return a * b

# 创建客服 Agent
agent = create_agent(
    model=model,
    tools=[calculator],
    system_prompt="""你是客服助手。
        特点：
        - 记住用户问题
        - 简洁回答
        - 使用工具计算
    """,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            max_tokens_before_summary=300  # 适合客服场景
        )
    ]
)

config = {"configurable": {"thread_id": "customer_123"}}

# 模拟客服对话
conversations = [
    "你好，我想咨询订单",
    "我的订单号是 12345",
    "帮我算一下 100 乘以 2 的优惠价",
    "谢谢"
]

for msg in conversations:
    print(f"\n客户: {msg}")
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    print(f"客服: {response['messages'][-1].content}")
    print("="*50)


客户: 你好，我想咨询订单
客服: 您好！欢迎咨询订单相关问题。

为了帮您更好地处理订单问题，我需要了解一些信息：

1. 您的订单号是多少？
2. 您遇到了什么问题？（比如：物流查询、退款申请、商品问题等）
3. 需要我帮您计算什么吗？（比如：运费、折扣金额、退款金额等）

请提供这些信息，我会尽力为您解答！

客户: 我的订单号是 12345
客服: 好的，我记下了您的订单号是 12345。

现在请告诉我您遇到了什么问题？比如：
- 物流查询
- 退款申请  
- 商品质量问题
- 订单状态查询
- 价格计算
- 其他问题

另外，如果您需要计算什么金额（比如运费、折扣、退款等），我也可以帮您计算。

客户: 帮我算一下 100 乘以 2 的优惠价
客服: 根据计算，100 乘以 2 的优惠价是 **200元**。

请问这个计算结果与您的订单 12345 有关吗？如果是的话，我可以帮您进一步处理订单问题。

客户: 谢谢
客服: 不客气！很高兴能帮到您。

如果您还有其他关于订单 12345 的问题，或者需要计算其他金额，随时告诉我。祝您购物愉快！
